# 08 — Multiprocessing, Process Pools, and the GIL

Processes can execute CPU-heavy work in parallel on multiple cores, with serialization and startup costs. Standalone Python scripts are safer for ProcessPoolExecutor than interactive notebook-defined functions, especially with spawn-based platforms.

*Advanced · 90–150 min*

## You will learn
- Explain CPU-bound vs I/O-bound work
- Understand inter-process serialization
- Choose a multiprocessing start method appropriately
- Keep process-pool examples in importable .py modules

## Examples

### 1. GIL and trade-offs

In common CPython builds, the GIL limits parallel execution of Python bytecode in threads. Processes have independent interpreters, but sending data across process boundaries takes time and memory. Do not assume a process pool always wins.

In [ ]:
import os
import sys
print("Python implementation:", sys.implementation.name)
print("CPU count:", os.cpu_count())
print("Multiprocessing requires importable worker functions.")

**Think:** Why might processes be slower for a task that completes in milliseconds?

### 2. CPU work to benchmark

Use a deterministic pure function as the workload. We keep the notebook serial and small so it runs on all supported platforms; the companion script in advanced/examples demonstrates actual process pools under a __main__ guard.

In [ ]:
def count_divisors(n):
    return sum(1 for candidate in range(1, n + 1) if n % candidate == 0)

inputs = [300, 400, 500]
print([count_divisors(n) for n in inputs])

**Think:** Which objects must be serializable when passing work to a process pool?

### 3. Process safety and main guards

On platforms using spawn, child interpreters import the main module. Starting a process at import time can recursively create processes. Keep the entry point under if __name__ == '__main__' and workers in importable modules.

In [ ]:
example = """
from concurrent.futures import ProcessPoolExecutor

def square(n):
    return n * n

if __name__ == '__main__':
    with ProcessPoolExecutor(max_workers=2) as pool:
        print(list(pool.map(square, [1, 2, 3])))
"""
print(example)

**Think:** Why does a notebook cell not always behave like an importable top-level module?

### 4. Chunking and cost models

Workers pay startup, scheduling, serialization, and result-collection costs. For many tiny jobs, batch items before handing them to workers; benchmark the whole pipeline rather than just a pure inner loop.

In [ ]:
import time
values = list(range(3000))
start = time.perf_counter()
result = sum(n * n for n in values)
elapsed = time.perf_counter() - start
print("result:", result, "measured_seconds:", round(elapsed, 6))

**Think:** What benchmark would justify adding processes instead of keeping the simple serial loop?

## Try it (optional)

Small open experiments to explore the topic. The **graded** exercises for this topic, with fill-in-the-blank code and automatic checks, are in the [workbook](../exercises/02_concurrency_resilience_exercises.ipynb).

**Exercise 1.** Run the companion process-pool example from a terminal and compare output.

In [ ]:
# Exercise 1
# TODO: Write code and tests here.

**Exercise 2.** Explain pickling requirements for worker arguments and return values.

In [ ]:
# Exercise 2
# TODO: Write code and tests here.

**Exercise 3.** Estimate when process overhead outweighs benefits.

In [ ]:
# Exercise 3
# TODO: Write code and tests here.

**Exercise 4.** Design a CPU-intensive pure function suitable for parallel map.

In [ ]:
# Exercise 4
# TODO: Write code and tests here.

## Common mistakes

- Do not define process-pool workers only inside interactive notebook cells.
- Processes add serialization and memory overhead.
- Always use the main guard for portable spawn-based execution.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** For CPU-bound pure-Python work, use…

   - **a)** asyncio
   - **b)** Processes
   - **c)** Threads

**2.** Data sent to a worker process must be…

   - **a)** A string
   - **b)** Global
   - **c)** Picklable

**3.** Why put ProcessPoolExecutor code under `if __name__ == '__main__':`?

   - **a)** Workers re-import the module; the guard stops them re-running it
   - **b)** Speed
   - **c)** Style only

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/08', {1: 'd604a269e1b8', 2: '1b51b24e890b', 3: '8fe1ad84a8e6'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [09 — Retry Policies, Rate Limiting, and Idempotency](09_retries_rate_limits_idempotency.ipynb)